# Semana 11 · OpenAI Agents SDK (experimento del estudiante)

Mismas secciones que la demo, con predicciones antes de ejecutar y TODOs. El notebook corre de principio a fin aunque los TODO queden sin llenar; llenarlos es la actividad.

## 0. Preparación

In [ ]:
# En Colab: instala el SDK de agentes y lo que usan las tools. En local ya está en .venv.
import importlib, subprocess, sys
for paquete, modulo in [("openai-agents", "agents"), ("openai", "openai"), ("pydantic", "pydantic"),
                        ("sentence-transformers", "sentence_transformers"), ("transformers[serving]", "transformers")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=False)
print("dependencias listas")

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
# Backend: llave de OpenAI si existe; si no, servidor local compatible (transformers serve).
# Copia reducida de course/shared/local_server.py para que el notebook sea autocontenido.
import os, shutil, subprocess, time, urllib.request

MODELO_LOCAL, PUERTO = "Qwen/Qwen2.5-1.5B-Instruct", 8765

def _vivo(raiz):
    try:
        with urllib.request.urlopen(f"{raiz}/health", timeout=2) as r: return r.status == 200
    except Exception: return False

def ensure_local_server(modelo=MODELO_LOCAL, puerto=PUERTO, espera_s=900):
    raiz = f"http://localhost:{puerto}"
    if _vivo(raiz): return raiz + "/v1"
    exe = shutil.which("transformers") or os.path.join(os.path.dirname(sys.executable), "transformers")
    subprocess.Popen([exe, "serve", "--port", str(puerto), modelo], stdout=open("/tmp/serve.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < espera_s:
        if _vivo(raiz): return raiz + "/v1"
        time.sleep(3)
    raise RuntimeError("el servidor local no arrancó; revisa /tmp/serve.log")

def backend_config():
    if os.environ.get("OPENAI_API_KEY") and not os.environ.get("OPENAI_BASE_URL"):
        return {"base_url": None, "api_key": os.environ["OPENAI_API_KEY"], "model": os.environ.get("OPENAI_MODEL", "gpt-5-mini"), "local": False}
    base = os.environ.get("OPENAI_BASE_URL") or ensure_local_server()
    return {"base_url": base, "api_key": "local", "model": MODELO_LOCAL, "local": True}

CFG = backend_config()
print("backend:", "servidor local (Chat Completions)" if CFG["local"] else "OpenAI (Responses API)", "| modelo:", CFG["model"])

In [ ]:
# Tools de negocio de Facturio: shared/facturio_tools.py. En local se importa; en Colab se descarga.
import sys, urllib.request
from pathlib import Path
CANDIDATOS = [Path("../../shared"), Path("../shared"), Path("shared")]
ruta = next((p for p in CANDIDATOS if (p / "facturio_tools.py").exists()), None)
if ruta is None:
    url = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/shared/facturio_tools.py"
    Path("shared").mkdir(exist_ok=True)
    urllib.request.urlretrieve(url, "shared/facturio_tools.py")
    ruta = Path("shared")
sys.path.insert(0, str(ruta.resolve()))
import facturio_tools as ft
FOLIO = list(ft.CARGOS)[0]
print("tools de negocio:", list(ft.TOOLS) if hasattr(ft, "TOOLS") else "cargadas", "| folio de ejemplo:", FOLIO)

In [ ]:
# Tracing de consola: un procesador propio que recibe los spans del SDK y los imprime como árbol.
# Con la llave, el SDK envía los spans al dashboard de OpenAI; aquí se sustituye el exportador.
from datetime import datetime
from agents.tracing import TracingProcessor, set_trace_processors, set_tracing_disabled

def _seg(span):
    try: return (datetime.fromisoformat(span.ended_at) - datetime.fromisoformat(span.started_at)).total_seconds()
    except Exception: return 0.0

class TracingConsola(TracingProcessor):
    def __init__(self): self.traces = []; self._actual = None
    def on_trace_start(self, trace): self._actual = {"nombre": trace.name, "spans": []}
    def on_trace_end(self, trace): self.traces.append(self._actual); self._actual = None
    def on_span_start(self, span): pass
    def on_span_end(self, span):
        d, tipo, usage = span.span_data, span.span_data.type, {}
        if tipo == "generation":
            usage = d.usage or {}; texto = f"generation  {_seg(span):5.2f} s  in={usage.get('input_tokens',0)} out={usage.get('output_tokens',0)}"
        elif tipo == "function": texto = f"function {d.name}({str(d.input)[:45]}) -> {str(d.output)[:45]}"
        elif tipo == "guardrail": texto = f"guardrail {d.name}  triggered={d.triggered}"
        elif tipo == "agent": texto = f"agent {d.name}  {_seg(span):5.2f} s"
        else: texto = f"{tipo}  {_seg(span):5.2f} s"
        if self._actual is not None:
            self._actual["spans"].append({"tipo": tipo, "id": span.span_id, "padre": span.parent_id, "usage": usage, "texto": texto, "nombre": getattr(d, "name", None), "seg": _seg(span)})
    def force_flush(self): pass
    def shutdown(self): pass
    def imprimir(self):
        t = self.traces[-1]; print("RUN", t["nombre"])
        hijos = {}
        for s in t["spans"]: hijos.setdefault(s["padre"], []).append(s)
        def rec(p, n):
            for i, s in enumerate(hijos.get(p, [])):
                print("    " * n + ("└── " if i == len(hijos[p]) - 1 else "├── ") + s["texto"]); rec(s["id"], n + 1)
        rec(None, 0)
    def resumen(self):
        t = self.traces[-1]; g = [s for s in t["spans"] if s["tipo"] == "generation"]
        return {"llamadas_modelo": len(g), "input_tokens": sum(s["usage"].get("input_tokens", 0) for s in g),
                "output_tokens": sum(s["usage"].get("output_tokens", 0) for s in g),
                "tool_calls": [s["nombre"] for s in t["spans"] if s["tipo"] == "function"],
                "latencia_s": round(max((s["seg"] for s in t["spans"] if s["tipo"] == "agent"), default=0), 2)}

set_tracing_disabled(False)
consola = TracingConsola()
set_trace_processors([consola])      # nada viaja a OpenAI; con la llave, quitar esta línea para usar el dashboard
print("tracing de consola activo")

In [ ]:
# El agente en el SDK. Cada pieza sustituye algo que ya se escribió a mano.
import json, asyncio
from pydantic import BaseModel, Field
from agents import (Agent, Runner, function_tool, ModelSettings, OpenAIChatCompletionsModel, input_guardrail, output_guardrail,
                    GuardrailFunctionOutput, InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered, SQLiteSession, StopAtTools)

def modelo():
    if CFG["local"]:                                     # el servidor local solo habla Chat Completions
        from openai import AsyncOpenAI
        return OpenAIChatCompletionsModel(model=CFG["model"], openai_client=AsyncOpenAI(base_url=CFG["base_url"], api_key="local"))
    return CFG["model"]                                  # con llave: Responses API por defecto

AJUSTES = ModelSettings(max_tokens=200, temperature=0)

@function_tool
def consultar_factura(folio: str) -> str:
    """Consulta una factura por folio (formato PAY- seguido de 8 dígitos). Devuelve estado, monto y fecha."""
    return json.dumps(ft.consultar_factura(folio), ensure_ascii=False)

@function_tool
def consultar_plan(usuario_id: str) -> str:
    """Consulta el plan de un usuario (u001 a u005): límites, uso actual y precio."""
    return json.dumps(ft.consultar_plan(usuario_id), ensure_ascii=False)

@function_tool
def buscar_documentacion(pregunta: str) -> str:
    """Busca en la documentación y políticas de Facturio. Úsala para dudas de políticas, límites y errores."""
    return json.dumps(ft.buscar_documentacion(pregunta, k=2), ensure_ascii=False)

@function_tool(needs_approval=True)                      # la corrida se detiene antes de ejecutar
def reembolsar(folio: str, monto: float, motivo: str) -> str:
    """Solicita un reembolso sobre un cargo. Tiene side effect: mueve dinero. Requiere aprobación humana."""
    ok, razon = ft.autorizar_reembolso(folio, monto)     # la regla de negocio sigue en la aplicación
    return json.dumps(ft.reembolsar(folio, monto, motivo) if ok else {"ejecutado": False, "razon": razon}, ensure_ascii=False)

@function_tool
def escalar_a_humano(motivo: str) -> str:
    """Pasa el caso a un agente humano cuando la política lo exige o falta información."""
    return json.dumps(ft.escalar_a_humano(motivo), ensure_ascii=False)

TOOLS = [consultar_factura, consultar_plan, buscar_documentacion, reembolsar, escalar_a_humano]
INSTRUCCIONES = ("Eres el asistente de la mesa de soporte de Facturio. Usa las tools cuando haga falta: consulta facturas y planes "
                 "antes de afirmar algo, busca la documentación para políticas, y escala a humano cuando falte información o la "
                 "política lo exija. Responde en español, breve.")

def construir_agente(tools=TOOLS, input_guardrails=(), output_guardrails=(), output_type=None, instrucciones=INSTRUCCIONES):
    return Agent(name="Soporte Facturio", instructions=instrucciones, tools=list(tools), model=modelo(), model_settings=AJUSTES,
                 input_guardrails=list(input_guardrails), output_guardrails=list(output_guardrails), output_type=output_type)

async def correr(agente, entrada, aprobar=None, session=None, max_turns=6):
    """Runner.run más el ciclo de aprobación: si una tool con needs_approval fue pedida, la aplicación decide y reanuda."""
    r = await Runner.run(agente, entrada, session=session, max_turns=max_turns)
    while getattr(r, "interruptions", None):
        estado = r.to_state()
        for item in r.interruptions:
            args = json.loads(item.raw_item.arguments or "{}")
            (estado.approve if (aprobar and aprobar(item.tool_name, args)) else estado.reject)(item)
        r = await Runner.run(agente, estado, session=session, max_turns=max_turns)
    return r

agente = construir_agente()
print("agente listo con", len(TOOLS), "tools")

## 1. Predicción: líneas y tokens

Antes de correr nada, escribe cuántas líneas crees que ahorra el SDK frente al loop manual de la semana 10 (unas 290 líneas completas, 24 en la versión reducida) y si gastará más, igual o menos tokens por tarea.

In [ ]:
MI_PREDICCION_LINEAS = None      # TODO: número de líneas que ahorra el SDK (puede ser negativo)
MI_PREDICCION_TOKENS = None      # TODO: "más", "igual" o "menos" tokens por tarea que el loop manual
MI_JUSTIFICACION = ""            # TODO: una línea
print("predicciones registradas:", MI_PREDICCION_LINEAS, MI_PREDICCION_TOKENS)

## 2. Primera corrida y árbol de tracing

In [ ]:
r = await correr(agente, f"¿Cuál es el estado de la factura {FOLIO}?")
print("Respuesta:", r.final_output)
consola.imprimir(); print(consola.resumen())

In [ ]:
# TODO: a partir del árbol de arriba, llena los cinco datos.
DATOS_DEL_ARBOL = {"llamadas_modelo": None, "tools": [], "tokens_entrada": None, "tokens_salida": None, "latencia_s": None}
print(DATOS_DEL_ARBOL)

## 3. Session

Predicción: ¿el modelo recordará el presupuesto en el tercer turno? ¿Cuántos items habrá en la sesión después de tres turnos?

In [ ]:
MI_PREDICCION_RECUERDA = None     # TODO: True o False
MI_PREDICCION_ITEMS = None        # TODO: entero
import tempfile, os
db = os.path.join(tempfile.gettempdir(), "facturio-w11-est.db")
if os.path.exists(db): os.remove(db)
sesion = SQLiteSession("u002", db)
for turno in ["Mi usuario es u002 y mi presupuesto máximo mensual es 500 pesos.",
              "Consulta mi plan y dime cuántas facturas al mes incluye.",
              "¿Cuál era mi presupuesto máximo?"]:
    r = await correr(agente, turno, session=sesion)
    print(f"USUARIO: {turno}\nAGENTE:  {r.final_output}\n")
print("items en la sesión:", len(await sesion.get_items()), "| predicción:", MI_PREDICCION_ITEMS)
os.remove(db)

In [ ]:
# TODO: responde en texto. ¿Qué política de contexto de la semana 9 aplica la sesión por sí sola? ¿Qué harías con 200 turnos?
RESPUESTA_SESSION = """
"""

## 4. Guardrails (actividad 2)

Escribe dos guardrails verificables: uno de entrada y uno de salida. Cada uno con un caso que lo dispara y uno que no. El de salida más valioso detecta una promesa sin acción: la respuesta afirma un reembolso y el trace no tiene `reembolsar` ejecutada.

In [ ]:
@input_guardrail
async def mi_guardrail_entrada(ctx, agent, entrada):
    texto = entrada if isinstance(entrada, str) else json.dumps(entrada, ensure_ascii=False)
    dispara = False                 # TODO: condición verificable sobre `texto`
    return GuardrailFunctionOutput(output_info={"motivo": "TODO"}, tripwire_triggered=dispara)

@output_guardrail
async def mi_guardrail_salida(ctx, agent, salida):
    texto = salida if isinstance(salida, str) else str(salida)
    dispara = False                 # TODO: condición verificable sobre `texto` (o sobre consola.resumen()["tool_calls"])
    return GuardrailFunctionOutput(output_info={"motivo": "TODO"}, tripwire_triggered=dispara)

CASOS = {"dispara_entrada": "TODO", "no_dispara_entrada": f"¿Cuál es el estado de la factura {FOLIO}?",
         "dispara_salida": "TODO", "no_dispara_salida": "Busca en la documentación el horario de soporte."}
agente_mio = construir_agente(input_guardrails=[mi_guardrail_entrada], output_guardrails=[mi_guardrail_salida])
for nombre, caso in CASOS.items():
    if caso == "TODO":
        print(f"{nombre:20s} sin caso todavía"); continue
    try:
        r = await correr(agente_mio, caso); print(f"{nombre:20s} OK  -> {str(r.final_output)[:60]}")
    except (InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered) as e:
        print(f"{nombre:20s} TRIPWIRE {e.guardrail_result.output.output_info}")

## 5. Aprobación humana

Predicción: si la aplicación rechaza el reembolso, ¿qué dirá el modelo? ¿Afirmará que lo hizo?

In [ ]:
MI_PREDICCION_RECHAZO = ""        # TODO: una línea
antes = len(getattr(ft, "REEMBOLSOS", []))
r = await correr(agente, f"Reembolsa 300 pesos de la factura {FOLIO} por cargo duplicado.", aprobar=lambda tool, args: False)
print("reembolsos registrados:", antes, "->", len(getattr(ft, "REEMBOLSOS", [])))
print("respuesta:", str(r.final_output)[:160])
consola.imprimir()
POLITICA_APROBACION = ""          # TODO: monto máximo, plazo, quién aprueba, qué se registra

## 6. Comparación medida (una corrida por tarea; sube CORRIDAS si el servidor lo permite)

In [ ]:
# El loop manual de la semana 10, reducido a lo esencial, con las MISMAS definiciones de tools que el SDK.
# Sirve de control: misma redacción, mismas instrucciones, mismo modelo. La diferencia medida es del runtime.
import time
from openai import OpenAI
cliente_chat = OpenAI(base_url=CFG["base_url"], api_key=CFG["api_key"] or "local", timeout=180)
DEFS = [{"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.params_json_schema}} for t in TOOLS]
EJECUTAR = {"consultar_factura": lambda a: ft.consultar_factura(**a), "consultar_plan": lambda a: ft.consultar_plan(**a),
            "buscar_documentacion": lambda a: ft.buscar_documentacion(a["pregunta"], k=2),
            "reembolsar": lambda a: ft.reembolsar(**a) if ft.autorizar_reembolso(a["folio"], a["monto"])[0] else {"ejecutado": False},
            "escalar_a_humano": lambda a: ft.escalar_a_humano(**a)}

def loop_manual(objetivo, max_iteraciones=6, aprobar=lambda n, a: True):
    mensajes = [{"role": "system", "content": INSTRUCCIONES}, {"role": "user", "content": objetivo}]
    calls, tin, tout, t0 = [], 0, 0, time.perf_counter()
    for it in range(1, max_iteraciones + 1):
        modelo_local = CFG["model"]
        resp = cliente_chat.chat.completions.create(model=modelo_local, messages=mensajes, tools=DEFS, tool_choice="auto", max_tokens=200, temperature=0)
        m, u = resp.choices[0].message, resp.usage
        tin += u.prompt_tokens; tout += u.completion_tokens
        if not m.tool_calls:                                                    # decidir: respuesta final
            return {"respuesta": m.content or "", "calls": calls, "pasos": it, "tokens": tin + tout, "latencia_s": round(time.perf_counter() - t0, 2)}
        mensajes.append({"role": "assistant", "content": m.content or "", "tool_calls": [{"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in m.tool_calls]})
        for tc in m.tool_calls:                                                 # actuar
            args = json.loads(tc.function.arguments or "{}"); calls.append(tc.function.name)
            if tc.function.name == "reembolsar" and not aprobar("reembolsar", args):
                salida = {"ejecutado": False, "razon": "rechazado"}
            else:
                salida = EJECUTAR[tc.function.name](args)
            mensajes.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(salida, ensure_ascii=False)[:900]})  # observar
    return {"respuesta": "", "calls": calls, "pasos": max_iteraciones, "tokens": tin + tout, "latencia_s": round(time.perf_counter() - t0, 2)}

LOC_MANUAL = 24   # líneas del loop de arriba, sin comentarios
print("loop manual listo (control del experimento)")

In [ ]:
# Mini gold set de la semana 10: tarea -> tools esperadas. El éxito se verifica en código.
GOLD = [
    ("t1", f"¿Cuál es el estado de la factura {FOLIO}?", {"consultar_factura"}),
    ("t2", "Consulta el plan del usuario u001 y dime su límite de proyectos y cuántos usa.", {"consultar_plan"}),
    ("t3", "Busca en la documentación cuántos días tengo para pedir un reembolso y dime el plazo.", {"buscar_documentacion"}),
    ("t4", f"Reembolsa 300 pesos de la factura {FOLIO} por cargo duplicado.", {"reembolsar"}),
    ("t5", "Quiero eliminar mi cuenta y todos mis datos hoy mismo; si no puedes hacerlo, escala a un humano.", {"escalar_a_humano", "buscar_documentacion"}),
    ("t6", "Busca en la documentación el horario de atención de soporte.", {"buscar_documentacion"}),
]
def exito(esperadas, calls): return bool(esperadas & set(calls))

In [ ]:
# Comparación: 1 corrida(s) por tarea con cada sistema. Mismo modelo, mismas tools, mismas instrucciones.
import statistics
CORRIDAS = 1
filas = []
for tid, tarea, esperadas in GOLD:
    for k in range(CORRIDAS):
        m = loop_manual(tarea)
        filas.append({"tarea": tid, "sistema": "manual", "exito": exito(esperadas, m["calls"]), "calls": m["calls"], "pasos": m["pasos"], "tokens": m["tokens"], "latencia_s": m["latencia_s"]})
        r = await correr(agente, tarea, aprobar=lambda n, a: True)
        s = consola.resumen()
        filas.append({"tarea": tid, "sistema": "sdk", "exito": exito(esperadas, s["tool_calls"]), "calls": s["tool_calls"], "pasos": s["llamadas_modelo"], "tokens": s["input_tokens"] + s["output_tokens"], "latencia_s": s["latencia_s"]})
        print(f"{tid} #{k+1}  manual: {m['calls']} {m['tokens']} tok {m['latencia_s']} s   |   sdk: {s['tool_calls']} {s['input_tokens']+s['output_tokens']} tok {s['latencia_s']} s")

def resumen(sistema):
    fs = [f for f in filas if f["sistema"] == sistema]
    variab = sum(1 for tid, *_ in GOLD if len({tuple(f["calls"]) for f in fs if f["tarea"] == tid}) > 1)
    return {"sistema": sistema, "éxito": round(sum(f["exito"] for f in fs) / len(fs), 2), "pasos": round(statistics.mean(f["pasos"] for f in fs), 2),
            "tokens": round(statistics.mean(f["tokens"] for f in fs)), "latencia_s": round(statistics.mean(f["latencia_s"] for f in fs), 2),
            "tareas con tools variables": f"{variab} de {len(GOLD)}"}
print(); print(resumen("manual")); print(resumen("sdk"))
print("líneas de código: loop manual", LOC_MANUAL, "| agente SDK (definición y correr): unas 30")

In [ ]:
# TODO: compara con tus predicciones de la sección 1 y explica cada diferencia por tarea (modelo, definiciones o runtime).
COMPARACION = """
"""
print("predicción líneas:", MI_PREDICCION_LINEAS, "| predicción tokens:", MI_PREDICCION_TOKENS)

## 7. Para la práctica 11

El starter de la práctica repite estas secciones con tres corridas por tarea, el formateador de salida estructurada y el reporte. Lleva contigo la tabla de la sección 6 y tus dos guardrails.